# A Unified Framework for Multi-Signal Uncertainty Clustering in Mathematical Reasoning
## Characterizing SFT vs RL Model Behaviors

### Abstract
This analysis presents a unified framework for quantifying uncertainty in Large Language Models (LLMs) by integrating four distinct signals: **Mechanistic Uncertainty**, **Semantic Entropy**, **Logit Gap**, and **Heuristic Scores**. Unlike prior surveys that analyze these metrics in isolation, we employ a clustering approach to identify distinct "reasoning phenotypes"—interpretable failure modes (e.g., "Confident Hallucinations" vs. "Reasoning Spirals")—and compare their prevalence across SFT and RL-trained models.

### Methodology & Definitions
1.  **Mechanistic Uncertainty:** Captures internal state uncertainty. We measure the entropy of specific "truth-sensitive" neuron activations (identified via `ActivationMonitor` in `uq_core.py`) during generation.
2.  **Heuristic Score:** A linguistic marker of uncertainty. We compute the semantic embedding distance (via `SentenceTransformer`) between the trace and uncertainty anchor words (e.g., *"maybe", "perhaps"*) versus certainty anchors (e.g., *"definitely", "proven"*).
3.  **Semantic Entropy (SE):** Measures inconsistency over meanings. We cluster sampled responses by semantic equivalence and compute the entropy of the resulting distribution. *Note: We utilize discrete SE here; future work will incorporate Kernel Language Entropy (KLE).*
4.  **Logit Gap:** The confidence margin between the top-1 and top-2 predicted tokens.

### Updates in this Version
*   **Validation:** Added AUROC, AUARC, and ECE (Expected Calibration Error) metrics.
*   **Baselines:** Added Trace Length (Reasoning Density) and Self-Consistency.
*   **Robustness:** Added Wilson Score Confidence Intervals for cluster accuracy estimates ($n<5$).

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
import os
import re
from IPython.display import display, HTML

# --- CONFIGURATION ---
DB_PATH = "db/results.sqlite"
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
PALETTE = sns.color_palette("viridis", as_cmap=True)

print("✅ Environment Setup Complete")

✅ Environment Setup Complete


## 1. Data Loading & Feature Engineering

We load results from the database, aggregating across experiments to facilitate SFT vs. RL comparisons. Derived features include:
*   **Trace Length:** Token count proxy (reasoning effort).
*   **Semantic Entropy:** Computed via grouping responses by `question_id`.

In [2]:
def load_data(db_path):
    if not os.path.exists(db_path):
        print(f"❌ Database not found at {db_path}")
        return None

    conn = sqlite3.connect(db_path)
    
    # Join Experiments to identify Model Types (SFT vs RL)
    query = """
    SELECT 
        e.name as exp_name,
        r.result_id, r.question_id_external, r.question_text, r.gold_answer, 
        r.predicted_answer, r.full_trace_text, r.is_correct,
        r.uq_avg_entropy as 'Entropy',
        r.uq_min_logit_gap as 'LogitGap',
        r.uq_heuristic_score as 'Heuristic',
        r.uq_mech_score as 'Mechanistic'
    FROM Results r
    LEFT JOIN Experiments e ON r.experiment_id = e.experiment_id
    """
    
    try:
        df = pd.read_sql(query, conn)
        conn.close()
        
        if df.empty: 
            print("⚠️ No data found.")
            return None

        # 1. Binary Correctness
        df['Correct'] = df['is_correct'].astype(int)
        
        # 2. Trace Length (Baseline)
        # Simple split as proxy for tokens
        df['TraceLength'] = df['full_trace_text'].fillna("").apply(lambda x: len(x.split()))
        
        # 3. Semantic Entropy & Self-Consistency (Group-wise)
        def calc_group_metrics(group):
            # Normalize answers (exact string match clustering)
            answers = group['predicted_answer'].fillna("").str.strip().str.lower()
            counts = answers.value_counts(normalize=True)
            
            # Semantic Entropy: -sum(p * log(p))
            se = -np.sum(counts * np.log(counts + 1e-10))
            
            # Self-Consistency: Probability of the majority answer
            sc = counts.iloc[0] if not counts.empty else 0.0
            
            return pd.Series({'SemanticEntropy': se, 'SelfConsistency': sc})

        if 'question_id_external' in df.columns:
            # Compute metrics per question group
            # Note: We group by exp_name too to avoid mixing SFT/RL samples for entropy calc
            metrics = df.groupby(['exp_name', 'question_id_external']).apply(calc_group_metrics).reset_index()
            df = df.merge(metrics, on=['exp_name', 'question_id_external'])
        else:
            df['SemanticEntropy'] = 0.0
            df['SelfConsistency'] = 1.0
            
        # 4. Infer Model Type
        def categorize_model(name):
            if name is None: return "General"
            name = name.lower()
            if 'rl' in name or 'ppo' in name: return 'RL'
            if 'sft' in name: return 'SFT'
            return 'General'
        
        df['ModelType'] = df['exp_name'].apply(categorize_model)
        
        # Handle NaNs
        feat_cols = ['Entropy', 'LogitGap', 'Heuristic', 'Mechanistic', 'SemanticEntropy', 'TraceLength', 'SelfConsistency']
        df[feat_cols] = df[feat_cols].fillna(0)

        print(f"Loaded {len(df)} traces across models: {df['ModelType'].unique()}")
        return df

    except Exception as e:
        print(f"Error: {e}")
        return None

df = load_data(DB_PATH)
if df is not None:
    display(df.head(3))

Loaded 440 traces across models: ['General']


/var/folders/q0/hw11rnjs0g1cnny7p5swsnv40000gn/T/ipykernel_54035/2854226768.py:54: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  metrics = df.groupby(['exp_name', 'question_id_external']).apply(calc_group_metrics).reset_index()


,exp_name,result_id,question_id_external,question_text,gold_answer,predicted_answer,full_trace_text,is_correct,Entropy,LogitGap,Heuristic,Mechanistic,Correct,TraceLength,SemanticEntropy,SelfConsistency,ModelType
0,HUV_Benchmark_Optimized,1,0,Natalia sold clips to 48 of her friends in Apr...,72,72,"In May, Natalia sold 1/2*48 = 24 clips.\nAlto...",1,0.071917,0.456573,0.407421,0.966309,1,20,0.562335,0.75,General
1,HUV_Benchmark_Optimized,2,0,Natalia sold clips to 48 of her friends in Apr...,72,72,To determine the total number of clips Natali...,1,0.030487,0.192776,0.482606,0.950987,1,126,0.562335,0.75,General
2,HUV_Benchmark_Optimized,3,0,Natalia sold clips to 48 of her friends in Apr...,72,72\nThe answer is: 72,"In April, Natalia sold 48 clips.\nIn May, she...",0,0.040688,0.243507,0.433654,0.962013,0,42,0.562335,0.75,General


## 2. Validation Metrics

We rigorously validate the uncertainty signals:
1.  **AUROC:** Ability to discriminate Correct vs Incorrect (Target: 1.0).
2.  **AUARC (Area Under Accuracy-Rejection Curve):** Utility for selective answering. Higher is better.
3.  **ECE (Expected Calibration Error):** We use Platt Scaling (Logistic Regression) to map scores to probabilities and measure calibration error.

In [7]:
def calculate_metrics(df, uncertainty_col, invert=False):
    """Calculates AUROC, AUARC, and ECE."""
    # Target: 1 = Error (we want Uncertainty to predict Error)
    y_true = 1 - df['Correct'].values
    scores = df[uncertainty_col].values
    
    # If metric is Confidence (e.g. LogitGap, SC), invert it to be Uncertainty
    if invert:
        scores = -scores
        
    # 1. AUROC
    try:
        auroc = roc_auc_score(y_true, scores)
    except:
        auroc = 0.5
        
    # 2. AUARC (Rejection Curve)
    # Sort by uncertainty (low to high for acceptance)
    sorted_indices = np.argsort(scores)
    sorted_correct = df['Correct'].values[sorted_indices]
    
    n = len(sorted_correct)
    accuracies = []
    rejection_rates = np.linspace(0, 0.95, 50)
    
    for r in rejection_rates:
        n_keep = int(n * (1 - r))
        if n_keep > 0:
            acc = np.mean(sorted_correct[:n_keep])
        else:
            acc = 1.0
        accuracies.append(acc)
    
    auarc = np.trapz(accuracies, rejection_rates)

    # 3. ECE (Calibration)
    # Fit Logistic Regression (Platt Scaling)
    lr = LogisticRegression()
    lr.fit(scores.reshape(-1, 1), y_true)
    probs = lr.predict_proba(scores.reshape(-1, 1))[:, 1]
    prob_true, prob_pred = calibration_curve(y_true, probs, n_bins=10)
    ece = np.mean(np.abs(prob_true - prob_pred))

    return auroc, auarc, ece

# Metrics to Validate
# Set invert=True if high value = Confidence (not Uncertainty)
metric_config = {
    'Entropy': False,
    'LogitGap': True,         # High Gap = Confident
    'Heuristic': False,       # High Heuristic = Uncertain (embedding distance from certainty)
    'Mechanistic': False,     # High Mech = Uncertain (Entropy)
    'SemanticEntropy': False,
    'TraceLength': False,      # Long Trace = Often Confident/Better Reasoning
    'SelfConsistency': True   # High SC = Confident
}

results = []
if df is not None:
    print(f"{'Metric':<20} | {'AUROC':<6} | {'AUARC':<6} | {'ECE':<6}")
    print("-"*50)
    for m, inv in metric_config.items():
        auroc, auarc, ece = calculate_metrics(df, m, invert=inv)
        results.append({'Metric': m, 'AUROC': auroc, 'AUARC': auarc, 'ECE': ece})
        print(f"{m:<20} | {auroc:.3f}  | {auarc:.3f}  | {ece:.3f}")

Metric               | AUROC  | AUARC  | ECE   
--------------------------------------------------
Entropy              | 0.605  | 0.724  | 0.088
LogitGap             | 0.601  | 0.731  | 0.102
Heuristic            | 0.530  | 0.696  | 0.028
Mechanistic          | 0.644  | 0.745  | 0.000
SemanticEntropy      | 0.738  | 0.788  | 0.032
TraceLength          | 0.601  | 0.715  | 0.102
SelfConsistency      | 0.736  | 0.787  | 0.033


/var/folders/q0/hw11rnjs0g1cnny7p5swsnv40000gn/T/ipykernel_54035/3155080974.py:34: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auarc = np.trapz(accuracies, rejection_rates)
/var/folders/q0/hw11rnjs0g1cnny7p5swsnv40000gn/T/ipykernel_54035/3155080974.py:34: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auarc = np.trapz(accuracies, rejection_rates)
/var/folders/q0/hw11rnjs0g1cnny7p5swsnv40000gn/T/ipykernel_54035/3155080974.py:34: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auarc = np.trapz(accuracies, rejection_rates)
/var/folders/q0/hw11rnjs0g1cnny7p5swsnv40000gn/T/ipykernel_54035/3155080974.py:34: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in

## 3. Clustering Analysis (The 13-Cluster Framework)

We standardize metrics and apply K-Means ($k=13$) to identify reasoning phenotypes. 

### Cluster Labeling & Statistics
We implement **Wilson Score Intervals** to provide 95% confidence intervals for cluster accuracy, ensuring that small clusters ($N<5$) are interpreted with caution.

In [4]:
def wilson_score_interval(successes, total, confidence=0.95):
    if total == 0: return 0, 0
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    p = successes / total
    denominator = 1 + z**2/total
    center = (p + z**2 / (2 * total)) / denominator
    std = np.sqrt((p * (1 - p) + z**2 / (4 * total)) / total) / denominator
    return max(0, center - z*std), min(1, center + z*std)

if df is not None:
    # 1. Prepare Features
    features = ['Entropy', 'LogitGap', 'Heuristic', 'Mechanistic', 'SemanticEntropy', 'TraceLength']
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[features])
    
    # 2. Clustering (k=13)
    kmeans = KMeans(n_clusters=13, random_state=42, n_init=10)
    df['Cluster'] = kmeans.fit_predict(X_scaled)
    
    # 3. Auto-Labeling via Z-Scores
    cluster_labels = {}
    global_means = df[features].mean()
    global_stds = df[features].std()
    
    for cid in range(13):
        sub = df[df['Cluster'] == cid]
        z_scores = (sub[features].mean() - global_means) / global_stds
        
        # Find distinguishing features (> 0.5 SD)
        high = z_scores[z_scores > 0.5].index.tolist()
        low = z_scores[z_scores < -0.5].index.tolist()
        
        label = "Avg"
        if high: label = "High " + "+".join([f[:4] for f in high[:2]])
        elif low: label = "Low " + "+".join([f[:4] for f in low[:2]])
        cluster_labels[cid] = label
        
    df['ClusterLabel'] = df['Cluster'].map(cluster_labels)
    
    # 4. Statistics Table
    stats_data = []
    for cid in sorted(df['Cluster'].unique()):
        sub = df[df['Cluster'] == cid]
        n = len(sub)
        corr = sub['Correct'].sum()
        acc = corr / n
        ci_low, ci_high = wilson_score_interval(corr, n)
        
        stats_data.append({
            'Cluster': cid,
            'Label': cluster_labels[cid],
            'Count': n,
            'Accuracy': acc,
            'CI_Lower': ci_low,
            'CI_Upper': ci_high,
            'TraceLen': sub['TraceLength'].mean()
        })
        
    stats_df = pd.DataFrame(stats_data)
    display(stats_df.style.format({'Accuracy': '{:.1%}', 'CI_Lower': '{:.1%}', 'CI_Upper': '{:.1%}', 'TraceLen': '{:.1f}'}))

,Cluster,Label,Count,Accuracy,CI_Lower,CI_Upper,TraceLen
0,0,High Heur,60,88.3%,77.8%,94.2%,184.7
1,1,High Sema,51,51.0%,37.7%,64.1%,164.9
2,2,Low Entr+Sema,61,70.5%,58.1%,80.4%,172.8
3,3,High Logi,39,92.3%,79.7%,97.3%,131.1
4,4,High Sema+Trac,30,63.3%,45.5%,78.1%,241.1
5,5,High Logi,31,90.3%,75.1%,96.7%,152.6
6,6,High Entr+Mech,19,26.3%,11.8%,48.8%,298.7
7,7,High Heur,40,80.0%,65.2%,89.5%,139.3
8,8,High Entr+Mech,11,9.1%,1.6%,37.7%,85.4
9,9,High Entr+Mech,37,86.5%,72.0%,94.1%,174.9


## 4. SFT vs RL Behavioral Comparison
We compare how different training methods distribute across the uncertainty clusters.

In [5]:
if df is not None and df['ModelType'].nunique() > 1:
    plt.figure(figsize=(12, 6))
    # Calculate proportion of each model's responses falling into each cluster
    props = df.groupby(['ModelType', 'ClusterLabel']).size().reset_index(name='counts')
    totals = df.groupby('ModelType').size().reset_index(name='total')
    props = props.merge(totals, on='ModelType')
    props['Proportion'] = props['counts'] / props['total']
    
    sns.barplot(data=props, x='Proportion', y='ClusterLabel', hue='ModelType', palette='muted')
    plt.title("Reasoning Phenotypes: SFT vs RL")
    plt.xlabel("Proportion of Model Responses")
    plt.tight_layout()
    plt.show()

## 5. Qualitative Dashboard
We generate the interactive HTML view to inspect traces, useful for analyzing "Confident Hallucinations" vs "Hedged Errors".

In [6]:
def generate_dashboard(df, stats_df):
    # Simple CSS for the dashboard
    css = """
    <style>
        .db-box { border: 1px solid #ddd; margin: 10px; padding: 15px; border-radius: 5px; }
        .db-header { display: flex; justify-content: space-between; font-weight: bold; background: #f8f9fa; padding: 10px; }
        .trace { background: #fafafa; font-family: monospace; font-size: 0.85em; padding: 10px; border-left: 3px solid #ccc; white-space: pre-wrap; }
        .correct { border-left-color: #28a745; } .incorrect { border-left-color: #dc3545; }
    </style>
    """
    html = [css, "<h2>Uncertainty Taxonomy Dashboard</h2>"]
    
    # Sort by Accuracy Ascending (Show failure modes first)
    sorted_stats = stats_df.sort_values('Accuracy')
    
    for _, row in sorted_stats.iterrows():
        cid = row['Cluster']
        acc = row['Accuracy']
        label = row['Label']
        count = row['Count']
        
        html.append(f"<div class='db-box'>")
        html.append(f"<div class='db-header'><span>Cluster {cid}: {label}</span> <span>Acc: {acc:.1%} (n={count})</span></div>")
        
        # Get samples
        samples = df[df['Cluster'] == cid].head(3)
        html.append("<details><summary>View Examples</summary>")
        for _, sample in samples.iterrows():
            status = "correct" if sample['Correct'] else "incorrect"
            trace = sample['full_trace_text'][:600] + "..." if len(sample['full_trace_text']) > 600 else sample['full_trace_text']
            html.append(f"<div class='trace {status}'><strong>Q:</strong> {sample['question_text']}<br><strong>Trace:</strong> {trace}</div>")
        html.append("</details></div>")
        
    return "\n".join(html)

if df is not None:
    dash = generate_dashboard(df, stats_df)
    display(HTML(dash))